# Bootstrap Bagging for Time-Series Forecasting

## Can residual resampling improve forecast accuracy?

This project investigates whether **bootstrap aggregation (bagging)** can improve forecasts of Motor Vehicle Stamp Duty revenue.

The workflow compares forecasts from the original time series against forecasts averaged across resampled versions of the series using:

- **ARIMA(1,1,1)**
- **ARIMA(1,1,0)**
- **VAR(5)** with `csi` and `hvi`
- **rolling or expanding backtests**
- **RMSE and MAE**
- **paired t-tests and Wilcoxon signed-rank tests**

### Why this notebook is structured this way

The original analysis was developed experimentally. This version reorganizes it into small reusable functions, removes repeated STL fits inside the bootstrap loop, keeps random-number generation reproducible, and prevents the seasonal-adjustment step from looking into the forecast period.

> **Reproducibility note:** the source Excel file is not included in this repository template. Put it in `data/Randomized_MV_Data.xlsx`, or change `DATA_PATH` below.

## 1. Imports and configuration

All project settings live in one place so the experiment can be changed without hunting through the notebook.

For a quick test run, try `N_BOOTSTRAPS = 10` and `BACKTEST_STEP_MONTHS = 12`.  
For the full experiment used in the original analysis, use `N_BOOTSTRAPS = 100` and `BACKTEST_STEP_MONTHS = 1`.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon
from statsmodels.tsa.api import VAR
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL

# -----------------------------
# Project configuration
# -----------------------------
DATA_PATH = Path("../data/Randomized_MV_Data.xlsx")

RANDOM_STATE = 42
N_BOOTSTRAPS = 100

SEASONAL_PERIOD = 12
STL_SEASONAL_SMOOTHER = 13

INITIAL_WINDOW_YEARS = 10
FORECAST_HORIZON_MONTHS = 12
BACKTEST_STEP_MONTHS = 1
WINDOW_TYPE = "rolling"  # "rolling" or "expanding"

ARIMA_ORDERS = {
    "ARIMA(1,1,1)": (1, 1, 1),
    "ARIMA(1,1,0)": (1, 1, 0),
}

VAR_LAG = 5

# Dates manually identified as unusual observations in the original analysis.
OUTLIER_DATES = pd.to_datetime([
    "2006-08-01",
    "2006-09-01",
    "2019-07-01",
    "2019-08-01",
    "2019-09-01",
    "2019-10-01",
])

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 2. Load and standardize the data

The original notebook used a machine-specific Windows path. Here the data path is relative to the repository, which makes the notebook portable.

Column names are normalized so trailing spaces such as `"Stamp Duty on Motor Vehicles "` do not break the analysis.

In [ ]:
def _normalize_column_name(name: str) -> str:
    """Convert a column name to a simple snake_case form."""
    return (
        str(name)
        .strip()
        .lower()
        .replace("&", "and")
        .replace("/", "_")
        .replace("-", "_")
        .replace(" ", "_")
    )


def load_data(path: Path) -> pd.DataFrame:
    """
    Load the monthly Motor Vehicle Stamp Duty dataset.

    Expected fields:
    - Date
    - Motor Vehicle Stamp Duty target, either `mvsd` or
      `Stamp Duty on Motor Vehicles`
    - `csi` and `hvi` for the VAR model
    """
    df = pd.read_excel(path)
    df.columns = [_normalize_column_name(c) for c in df.columns]

    if "date" not in df.columns:
        raise ValueError("Expected a 'Date' column in the input file.")

    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    df.index = df.index.to_period("M").to_timestamp("MS")
    df = df[~df.index.duplicated(keep="first")]

    target_candidates = [
        "mvsd",
        "stamp_duty_on_motor_vehicles",
    ]
    target = next((c for c in target_candidates if c in df.columns), None)

    if target is None:
        raise ValueError(
            "Could not find the Motor Vehicle Stamp Duty column. "
            f"Available columns: {list(df.columns)}"
        )

    if target != "mvsd":
        df = df.rename(columns={target: "mvsd"})

    df = df.asfreq("MS")

    if df["mvsd"].isna().any():
        missing = int(df["mvsd"].isna().sum())
        raise ValueError(
            f"The target contains {missing} missing monthly observations. "
            "Resolve these before running the forecasting experiment."
        )

    return df


if DATA_PATH.exists():
    data = load_data(DATA_PATH)
    display(data.head())
    print(f"Rows: {len(data):,}")
    print(f"Date range: {data.index.min():%Y-%m} to {data.index.max():%Y-%m}")
else:
    data = None
    print(
        f"Data file not found at: {DATA_PATH}\n"
        "Add the Excel file there, or edit DATA_PATH in the configuration cell."
    )

## 3. Optional outlier correction

The original analysis identified six dates as unusual and replaced their STL residuals with zero. The function below preserves that idea while isolating it in one clearly documented preprocessing step.

This uses the full series to reconstruct **pre-identified** outliers. For a strict real-time forecasting study, either justify these dates as known data corrections or move outlier detection inside each training window.

In [ ]:
def clean_known_outliers(
    series: pd.Series,
    outlier_dates,
    period: int = 12,
    seasonal_smoother: int = 13,
) -> tuple[pd.Series, object]:
    """
    Reconstruct selected observations with STL residual set to zero.

    Returns
    -------
    cleaned_series : pd.Series
        Trend + seasonal + adjusted residual.
    decomposition : statsmodels STL result
        STL result from the original series.
    """
    series = series.astype(float).copy()

    decomposition = STL(
        series,
        period=period,
        seasonal=seasonal_smoother,
    ).fit()

    residual = decomposition.resid.copy()
    valid_dates = series.index.intersection(pd.DatetimeIndex(outlier_dates))
    residual.loc[valid_dates] = 0.0

    cleaned = decomposition.trend + decomposition.seasonal + residual
    cleaned.name = "mvsd_clean"

    return cleaned, decomposition


if data is not None:
    data = data.copy()
    data["mvsd_clean"], full_stl = clean_known_outliers(
        data["mvsd"],
        OUTLIER_DATES,
        period=SEASONAL_PERIOD,
        seasonal_smoother=STL_SEASONAL_SMOOTHER,
    )

    ax = data[["mvsd", "mvsd_clean"]].plot(
        figsize=(12, 5),
        title="Motor Vehicle Stamp Duty: original vs cleaned series",
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Revenue")
    ax.legend(["Original", "Cleaned"])
    plt.tight_layout()
    plt.show()

## 4. STL decomposition

STL separates the series into trend, seasonal, and residual components. The bootstrap step resamples the **residual** component while preserving the estimated trend.

A key change from the original implementation is that STL is fitted **once per training window**, rather than once for every bootstrap sample.

In [ ]:
def decompose_training_series(
    series: pd.Series,
    period: int = 12,
    seasonal_smoother: int = 13,
):
    """Fit STL to one training series."""
    return STL(
        series.astype(float),
        period=period,
        seasonal=seasonal_smoother,
    ).fit()


def repeat_last_season(
    seasonal_component: pd.Series,
    horizon: int,
    period: int = 12,
) -> np.ndarray:
    """Extend the most recent seasonal cycle over the forecast horizon."""
    pattern = seasonal_component.iloc[-period:].to_numpy()
    repeats = int(np.ceil(horizon / period))
    return np.tile(pattern, repeats)[:horizon]


if data is not None:
    stl_example = decompose_training_series(
        data["mvsd_clean"],
        period=SEASONAL_PERIOD,
        seasonal_smoother=STL_SEASONAL_SMOOTHER,
    )
    stl_example.plot()
    plt.suptitle("STL decomposition of cleaned Motor Vehicle Stamp Duty", y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Residual resampling

The original project randomly **permuted** STL residuals. The function below keeps that as the default so the refactor remains close to the original experiment.

If you want a classical residual bootstrap with replacement, set `method="replacement"`.

In [ ]:
def make_bootstrap_deseasonalized_series(
    decomposition,
    n_bootstraps: int,
    rng: np.random.Generator,
    method: str = "permutation",
) -> pd.DataFrame:
    """
    Create bootstrapped deseasonalized training series.

    Each sample is:
        trend + resampled residual

    This avoids performing another STL decomposition on every bootstrap sample.
    """
    trend = decomposition.trend.to_numpy()
    residual = decomposition.resid.to_numpy()
    n_obs = len(residual)

    if method not in {"permutation", "replacement"}:
        raise ValueError("method must be 'permutation' or 'replacement'")

    samples = np.empty((n_obs, n_bootstraps), dtype=float)

    for i in range(n_bootstraps):
        if method == "permutation":
            sampled_residual = rng.permutation(residual)
        else:
            sampled_residual = rng.choice(residual, size=n_obs, replace=True)

        samples[:, i] = trend + sampled_residual

    return pd.DataFrame(
        samples,
        index=decomposition.trend.index,
        columns=[f"bootstrap_{i:03d}" for i in range(1, n_bootstraps + 1)],
    )

## 6. Forecasting helpers

In [ ]:
def forecast_arima(
    series: pd.Series,
    order: tuple[int, int, int],
    horizon: int,
) -> np.ndarray:
    """Fit one ARIMA model and return a numeric forecast."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = ARIMA(
            series,
            order=order,
            enforce_stationarity=True,
            enforce_invertibility=True,
        ).fit()

    return np.asarray(fitted.forecast(steps=horizon), dtype=float)


def forecast_bootstrap_arima(
    bootstrapped_series: pd.DataFrame,
    order: tuple[int, int, int],
    horizon: int,
) -> np.ndarray:
    """Fit ARIMA to every bootstrap sample and average the forecasts."""
    forecasts = [
        forecast_arima(bootstrapped_series[col], order, horizon)
        for col in bootstrapped_series.columns
    ]
    return np.mean(np.vstack(forecasts), axis=0)


def forecast_var(
    train_df: pd.DataFrame,
    target_col: str,
    horizon: int,
    lag: int = 5,
) -> np.ndarray | None:
    """
    Forecast the target from a VAR model.
    Returns None if there are not enough complete observations.
    """
    complete = train_df.dropna()

    if len(complete) <= lag + 5:
        return None

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = VAR(complete).fit(lag)

    forecast = fitted.forecast(
        complete.to_numpy()[-fitted.k_ar:],
        steps=horizon,
    )

    target_position = complete.columns.get_loc(target_col)
    return forecast[:, target_position]


def forecast_bootstrap_var(
    bootstrap_target: pd.DataFrame,
    covariates: pd.DataFrame,
    horizon: int,
    lag: int = 5,
) -> np.ndarray | None:
    """Fit one VAR per bootstrap target series and average target forecasts."""
    forecasts = []

    for col in bootstrap_target.columns:
        candidate = pd.concat(
            [
                bootstrap_target[col].rename("mvsd_ds"),
                covariates,
            ],
            axis=1,
        )

        forecast = forecast_var(
            candidate,
            target_col="mvsd_ds",
            horizon=horizon,
            lag=lag,
        )

        if forecast is not None:
            forecasts.append(forecast)

    if not forecasts:
        return None

    return np.mean(np.vstack(forecasts), axis=0)

## 7. Rolling / expanding backtest

Each forecast is produced using **training data only**. Future observations are used only for evaluation.

For every window:

1. Fit STL to the training target.
2. Remove seasonality from the training target.
3. Create resampled deseasonalized training series.
4. Fit original and bagged ARIMA forecasts.
5. Fit original and bagged VAR forecasts when `csi` and `hvi` are available.
6. Repeat the most recent seasonal cycle and add it back to each forecast.
7. Compare forecasts with the actual future target.

In [ ]:
def _rmse(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.sqrt(np.mean((forecast - actual) ** 2)))


def _mae(actual: np.ndarray, forecast: np.ndarray) -> float:
    return float(np.mean(np.abs(forecast - actual)))


def run_backtest(
    df: pd.DataFrame,
    target_col: str = "mvsd_clean",
    n_bootstraps: int = 100,
    initial_window_years: int = 10,
    horizon: int = 12,
    step_months: int = 1,
    window_type: str = "rolling",
    random_state: int = 42,
    bootstrap_method: str = "permutation",
    var_columns: tuple[str, str] = ("csi", "hvi"),
    var_lag: int = 5,
    verbose: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run rolling or expanding out-of-sample forecasting backtests.

    Returns
    -------
    window_metrics : pd.DataFrame
        One row per window/model/method with RMSE and MAE.
    point_forecasts : pd.DataFrame
        Forecast and actual value for every forecasted month.
    """
    if window_type not in {"rolling", "expanding"}:
        raise ValueError("window_type must be 'rolling' or 'expanding'")

    df = df.sort_index().copy()
    initial_months = initial_window_years * 12
    rng = np.random.default_rng(random_state)

    if len(df) < initial_months + horizon:
        raise ValueError("Not enough observations for the requested backtest.")

    metric_rows = []
    forecast_rows = []
    window_number = 0

    for train_end in range(
        initial_months,
        len(df) - horizon + 1,
        step_months,
    ):
        window_number += 1

        train_start = 0 if window_type == "expanding" else train_end - initial_months

        train = df.iloc[train_start:train_end].copy()
        test = df.iloc[train_end:train_end + horizon].copy()

        train_target = train[target_col].astype(float)
        actual = test[target_col].to_numpy(dtype=float)

        # STL sees training data only.
        decomposition = decompose_training_series(
            train_target,
            period=SEASONAL_PERIOD,
            seasonal_smoother=STL_SEASONAL_SMOOTHER,
        )

        train_deseasonalized = (
            train_target - decomposition.seasonal
        ).rename("mvsd_ds")

        seasonal_future = repeat_last_season(
            decomposition.seasonal,
            horizon=horizon,
            period=SEASONAL_PERIOD,
        )

        bootstrap_ds = make_bootstrap_deseasonalized_series(
            decomposition,
            n_bootstraps=n_bootstraps,
            rng=rng,
            method=bootstrap_method,
        )

        forecasts = {}

        # ARIMA models
        for model_name, order in ARIMA_ORDERS.items():
            original_ds_forecast = forecast_arima(
                train_deseasonalized,
                order=order,
                horizon=horizon,
            )
            bagged_ds_forecast = forecast_bootstrap_arima(
                bootstrap_ds,
                order=order,
                horizon=horizon,
            )

            forecasts[(model_name, "Original")] = (
                original_ds_forecast + seasonal_future
            )
            forecasts[(model_name, "Bootstrapped")] = (
                bagged_ds_forecast + seasonal_future
            )

        # VAR model, if the required covariates exist.
        if all(col in train.columns for col in var_columns):
            covariates = train[list(var_columns)]

            var_original_train = pd.concat(
                [train_deseasonalized, covariates],
                axis=1,
            )

            original_var_ds = forecast_var(
                var_original_train,
                target_col="mvsd_ds",
                horizon=horizon,
                lag=var_lag,
            )

            bagged_var_ds = forecast_bootstrap_var(
                bootstrap_target=bootstrap_ds,
                covariates=covariates,
                horizon=horizon,
                lag=var_lag,
            )

            if original_var_ds is not None:
                forecasts[("VAR(5)", "Original")] = (
                    original_var_ds + seasonal_future
                )

            if bagged_var_ds is not None:
                forecasts[("VAR(5)", "Bootstrapped")] = (
                    bagged_var_ds + seasonal_future
                )

        for (model_name, method), forecast in forecasts.items():
            metric_rows.append({
                "window": window_number,
                "train_start": train.index.min(),
                "train_end": train.index.max(),
                "test_start": test.index.min(),
                "test_end": test.index.max(),
                "model": model_name,
                "method": method,
                "rmse": _rmse(actual, forecast),
                "mae": _mae(actual, forecast),
            })

            for date, actual_value, forecast_value in zip(
                test.index,
                actual,
                forecast,
            ):
                forecast_rows.append({
                    "window": window_number,
                    "date": date,
                    "model": model_name,
                    "method": method,
                    "actual": actual_value,
                    "forecast": float(forecast_value),
                })

        if verbose and (window_number == 1 or window_number % 12 == 0):
            print(
                f"Completed window {window_number}: "
                f"train through {train.index.max():%Y-%m}, "
                f"forecast through {test.index.max():%Y-%m}"
            )

    return pd.DataFrame(metric_rows), pd.DataFrame(forecast_rows)

## 8. Run the experiment

The full monthly backtest can be computationally expensive because each window fits many ARIMA and VAR models. The code below uses the configuration at the top of the notebook.

In [ ]:
if data is not None:
    window_metrics, point_forecasts = run_backtest(
        data,
        target_col="mvsd_clean",
        n_bootstraps=N_BOOTSTRAPS,
        initial_window_years=INITIAL_WINDOW_YEARS,
        horizon=FORECAST_HORIZON_MONTHS,
        step_months=BACKTEST_STEP_MONTHS,
        window_type=WINDOW_TYPE,
        random_state=RANDOM_STATE,
        bootstrap_method="permutation",
        var_lag=VAR_LAG,
        verbose=True,
    )

    display(window_metrics.head())
    print(f"Metric rows: {len(window_metrics):,}")
else:
    window_metrics = None
    point_forecasts = None

## 9. Summarize forecast accuracy

In [ ]:
def summarize_results(window_metrics: pd.DataFrame) -> pd.DataFrame:
    """
    Compare mean RMSE and MAE for original vs bootstrapped forecasts.

    Positive improvement percentages mean the bootstrapped forecast had
    lower error.
    """
    grouped = (
        window_metrics
        .groupby(["model", "method"])[["rmse", "mae"]]
        .mean()
        .unstack("method")
    )

    rows = []

    for model in grouped.index:
        needed = {
            ("rmse", "Original"),
            ("rmse", "Bootstrapped"),
            ("mae", "Original"),
            ("mae", "Bootstrapped"),
        }
        if not needed.issubset(grouped.columns):
            continue

        original_rmse = grouped.loc[model, ("rmse", "Original")]
        boot_rmse = grouped.loc[model, ("rmse", "Bootstrapped")]
        original_mae = grouped.loc[model, ("mae", "Original")]
        boot_mae = grouped.loc[model, ("mae", "Bootstrapped")]

        rows.append({
            "model": model,
            "original_rmse": original_rmse,
            "bootstrapped_rmse": boot_rmse,
            "rmse_improvement_pct": 100 * (original_rmse - boot_rmse) / original_rmse,
            "original_mae": original_mae,
            "bootstrapped_mae": boot_mae,
            "mae_improvement_pct": 100 * (original_mae - boot_mae) / original_mae,
        })

    return pd.DataFrame(rows).sort_values("model").reset_index(drop=True)


if window_metrics is not None:
    summary = summarize_results(window_metrics)
    display(summary)

## 10. Plot RMSE across backtest windows

In [ ]:
def plot_rmse_by_window(
    window_metrics: pd.DataFrame,
    model: str,
) -> None:
    """Plot original and bootstrapped RMSE for one model."""
    subset = window_metrics.loc[window_metrics["model"] == model]

    if subset.empty:
        print(f"No results available for {model}.")
        return

    pivot = subset.pivot(
        index="window",
        columns="method",
        values="rmse",
    )

    ax = pivot.plot(
        figsize=(11, 4.5),
        title=f"RMSE across backtest windows: {model}",
    )
    ax.set_xlabel("Window")
    ax.set_ylabel("RMSE")
    plt.tight_layout()
    plt.show()


if window_metrics is not None:
    for model_name in ["ARIMA(1,1,1)", "ARIMA(1,1,0)", "VAR(5)"]:
        plot_rmse_by_window(window_metrics, model_name)

## 11. Paired statistical tests

In [ ]:
def paired_tests(window_metrics: pd.DataFrame) -> pd.DataFrame:
    """
    Compare bootstrapped vs original RMSE within the same backtest windows.

    H0: there is no paired difference in window-level RMSE.
    """
    rows = []

    for model, group in window_metrics.groupby("model"):
        paired = (
            group
            .pivot(index="window", columns="method", values="rmse")
            .dropna()
        )

        if not {"Original", "Bootstrapped"}.issubset(paired.columns):
            continue

        original = paired["Original"].to_numpy()
        boot = paired["Bootstrapped"].to_numpy()

        t_stat, p_value_t = ttest_rel(boot, original)

        try:
            w_stat, p_value_w = wilcoxon(boot, original)
        except ValueError:
            w_stat, p_value_w = np.nan, np.nan

        rows.append({
            "model": model,
            "n_windows": len(paired),
            "mean_rmse_difference_boot_minus_original": np.mean(boot - original),
            "t_stat": t_stat,
            "p_value_t": p_value_t,
            "w_stat": w_stat,
            "p_value_w": p_value_w,
        })

    return pd.DataFrame(rows)


if window_metrics is not None:
    test_results = paired_tests(window_metrics)
    display(test_results)

## 12. Reference results from the original run

Before this refactor, the original notebook reported approximately:

| Model | Bootstrapped RMSE | Original RMSE |
|---|---:|---:|
| ARIMA(1,1,1) | 3.289M | 3.170M |
| ARIMA(1,1,0) | 3.291M | 3.345M |
| VAR(5) | 39.97M | 44.76M |

The strongest improvement appeared for **VAR(5)**. The original paired tests did not show a statistically significant improvement for either ARIMA specification, while the VAR comparison was statistically significant.

> **Important:** treat these numbers as reference results from the earlier implementation, not as outputs from this cleaned notebook. The refactor removes look-ahead from seasonal adjustment and may therefore change the exact values when rerun.

## 13. Conclusion and limitations

The project tests a useful question rather than assuming bagging will always win: **does residual resampling actually improve out-of-sample forecasts?**

The original results suggested that the answer depends on the forecasting model. Bagging had little or mixed benefit for the two ARIMA specifications, while the VAR model showed a larger reduction in RMSE.

### Limitations

- The outlier dates were manually selected and should be justified in the project write-up.
- Residual permutation preserves the residual values but is not the same as classical bootstrap sampling with replacement.
- ARIMA and VAR hyperparameters are fixed rather than selected within each training window.
- A one-month backtest step produces many overlapping 12-month forecast windows, so successive RMSE observations are not fully independent.
- The cleaned notebook should be rerun from start to finish before publishing final numerical claims.

### Possible extensions

- Compare permutation resampling with sampling **with replacement**.
- Add a naïve seasonal benchmark.
- Tune ARIMA orders using training data only.
- Explore block bootstrap methods if residual dependence remains.
- Save the final summary tables and plots to the `figures/` and `results/` folders.